# End-to-End Causal ML & Uplift Modeling Project

This notebook implements the concepts discussed in the companion guide (`causal_inference_guide.md`). We will use the **Hillstrom Email Marketing Dataset** to estimate the Conditional Average Treatment Effect (CATE) of different email campaigns on customer spend.

### Goal
Predict which customers should receive an email promotion to maximize incremental revenue (uplift), while ignoring customers who will buy anyway ("Sure Things") and customers who will never buy ("Lost Causes").

### Table of Contents
1. Setup & Data Loading
2. Data Preprocessing & EDA
3. Model Building: S-Learner, T-Learner, X-Learner
4. Model Building: R-Learner (CausalML)
5. Evaluation (Uplift & Qini Curves)
6. Business Decision Framework (Profitability)


In [ ]:
# ==========================================
# 1. Setup & Imports
# ==========================================
# !pip install scikit-uplift causalml lightgbm pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modeling
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.model_selection import train_test_split

# Uplift packages
from sklift.datasets import fetch_hillstrom
from sklift.models import SoloModel, TwoModels # S and T learners
from sklift.metrics import qini_auc_score, uplift_auc_score
from sklift.viz import plot_qini_curve, plot_uplift_curve
from causalml.inference.meta import BaseXRegressor, BaseRRegressor


### Data Loading
We use the **Hillstrom MineThatData** challenge dataset. It contains 64,000 customers randomly assigned to:
- Women's E-Mail
- Men's E-Mail
- No E-Mail (Control)

For simplicity, we will combine the Women's and Men's email groups into a single **Treatment = 1** group, and No E-Mail as **Control = 0**. Our target variable is `spend` (continuous revenue amount).

In [ ]:
# ==========================================
# 2. Data Loading & Preprocessing
# ==========================================

# Fetch data
dataset = fetch_hillstrom(target_col='spend')
X, y, treatment = dataset.data, dataset.target, dataset.treatment

# Convert treatment to binary: Any Email = 1, No Email (Control) = 0
treatment = treatment.map({'Mens E-Mail': 1, 'Womens E-Mail': 1, 'No E-Mail': 0})

# Combine all into one dataframe for easy feature engineering
df = X.copy()
df['treatment'] = treatment
df['spend'] = y

print(f"Dataset shape: {df.shape}")
display(df.head())

# A quick look at the average spend between Control and Treatment
print("\nAverage spend by group:")
print(df.groupby('treatment')['spend'].mean())


In [ ]:
# Feature Engineering: Encode categorical variables (OHE / Dummies)
categorical_cols = ['zip_code', 'channel', 'history_segment']
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Train/Test Split
X_train, X_val, y_train, y_val, trmnt_train, trmnt_val = train_test_split(
    X_encoded, y, treatment, test_size=0.3, random_state=42, stratify=treatment
)

print(f"Training instances: {len(X_train)}")
print(f"Validation instances: {len(X_val)}")


### Meta-Learner Implementations
Here, we initialize our base ML algorithms. We will use `LightGBM` as it handles non-linearities well, is extremely fast, and robust to outliers.

**1. S-Learner (Single Model)**: Learns $E[Y | X, T]$. Treatment is just a feature.
**2. T-Learner (Two Models)**: Learns $E[Y|X, T=0]$ and $E[Y|X, T=1]$ separately.
**3. X-Learner (Cross/Extrapolated)**: Learns control/treatment responses, imputes counterfactuals, and trains two final estimators.
**4. R-Learner (Residual)**: Learns base outcome $E[Y|X]$, propensity $E[T|X]$, computes residuals, and fits on residuals to isolate pure causal effect.

In [ ]:
# ==========================================
# 3. Model Building: S, T, and X Learners
# ==========================================

# Base model config
base_lgbm = LGBMRegressor(max_depth=4, n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)

print("Training S-Learner...")
s_learner = SoloModel(base_lgbm)
s_learner = s_learner.fit(X_train, y_train, trmnt_train)
uplift_s = s_learner.predict(X_val)

print("Training T-Learner...")
t_learner = TwoModels(
    estimator_trmnt=LGBMRegressor(max_depth=4, n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1),
    estimator_ctrl=LGBMRegressor(max_depth=4, n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1),
    method='vanilla'
)
t_learner = t_learner.fit(X_train, y_train, trmnt_train)
uplift_t = t_learner.predict(X_val)

print("Training X-Learner...")
x_learner = BaseXRegressor(
    learner=LGBMRegressor(max_depth=4, n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1),
    control_name=0
)
x_learner.fit(X=X_train.values, treatment=trmnt_train.values, y=y_train.values)
# The predict method returns a matrix of shape (n_samples, n_treatments). Since we only have 1 treatment group (vs 1 control), we flatten it.
uplift_x = x_learner.predict(X_val.values).flatten()


In [ ]:
# ==========================================
# 4. Model Building: R-Learner
# ==========================================
print("Training R-Learner...")
r_learner = BaseRRegressor(
    outcome_learner=LGBMRegressor(max_depth=5, n_estimators=150, learning_rate=0.05, random_state=42, verbose=-1),
    effect_learner=LGBMRegressor(max_depth=4, n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1),
    control_name=0
)

# Fitting
r_learner.fit(
    X=X_train.values, 
    treatment=trmnt_train.values, 
    y=y_train.values
)

# CATE Predictions
uplift_r = r_learner.predict(X_val.values).flatten()
print("All models trained successfully.")


### Evaluation
We evaluate uplift models using **Qini Curves** and **Uplift Curves**. 
Since standard metrics (RMSE) fail due to the lack of ground-truth counterfactuals, these curves sort users by their predicted uplift and measure the *incremental response rate* cumulatively.

In [ ]:
# ==========================================
# 5. Evaluation (Qini curves / AUUC)
# ==========================================

# Compile predictions into a DataFrame for visualization
df_preds = pd.DataFrame({
    'S-Learner': uplift_s,
    'T-Learner': uplift_t,
    'X-Learner': uplift_x,
    'R-Learner': uplift_r
}, index=X_val.index)

models_to_evaluate = ['S-Learner', 'T-Learner', 'X-Learner', 'R-Learner']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Qini curves
for model in models_to_evaluate:
    plot_qini_curve(y_val, df_preds[model], trmnt_val, perfect=False, name=model, ax=axes[0])
axes[0].set_title('Qini Curves')

# Plot Uplift curves
for model in models_to_evaluate:
    plot_uplift_curve(y_val, df_preds[model], trmnt_val, perfect=False, name=model, ax=axes[1])
axes[1].set_title('Uplift Curves')

plt.tight_layout()
plt.show()

# Print final metrics (Area Under Qini)
print("Area Under Qini Curve (Qini Coefficient):")
for model in models_to_evaluate:
    score = qini_auc_score(y_val, df_preds[model], trmnt_val)
    print(f"{model}: {score:.4f}")


### Feature Importance & Business Application
How do we actually use this to make money?
We calculate the expected profit per customer if we send an email vs. if we don't. 
If the `Expected Uplift In Revenue` > `Cost of Sending Email + Promotion Margin Loss`, we target them!

In [ ]:
# ==========================================
# 6. Business Decision Framework
# ==========================================

# Let's say sending the email costs $0.10, and it offers a fixed 20% discount on the margin.
# For simplicity, let's just assume a flat cost per treatment of $0.50 per sent email.
COST_PER_EMAIL = 0.50 

# Add the best model's predictions (say, R-Learner) to our validation set
val_results = X_val.copy()
val_results['predicted_cate'] = df_preds['R-Learner']

# Profitability Logic: Only treat users whose expected causal uplift > cost to treat
val_results['profitable_to_treat'] = val_results['predicted_cate'] > COST_PER_EMAIL

num_to_target = val_results['profitable_to_treat'].sum()
total_users = len(val_results)

print(f"Under causal modeling, we should only target {num_to_target} out of {total_users} customers.")
print(f"This avoids wasting {COST_PER_EMAIL} on {total_users - num_to_target} 'Sure Things' and 'Lost Causes'!")

# Let's inspect the top 5 customers we DEFINITELY should target
display(val_results[val_results['profitable_to_treat']].sort_values(by='predicted_cate', ascending=False).head())
